In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spectrum_kvco_S5_20261005_140920.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spectrum_kvco_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
runs = pd.read_csv(RESULT_CSV).reset_index(drop=True)
for col in ("time_jitter_noise_ps", "time_jitter_spur_ps", "n_spurs", "spurs", "carrier_dbm"):  # older runs
    if col not in runs:
        runs[col] = np.nan
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_vco", "kvco_target", "kind"])
offsets = [c[len("L_vco_"):] for c in runs.columns if c.startswith("L_vco_")]

# Only locked settings: without lock there is no carrier and the measured "phase noise" is the analyzer noise
unlocked = runs[runs["locked"].fillna(0) != 1]
runs = runs[runs["locked"].fillna(0) == 1].reset_index(drop=True)
print(f"{RESULT_CSV.name}: {len(runs) + len(unlocked)} settings, {len(runs)} locked "
      f"({(runs['status'] == 'ok').sum()} measured ok); traces: {TRACE_CSV.name if len(traces) else 'none'}")
if len(unlocked):
    print("excluded (not locked): " + ", ".join(
        f"|Kvco| {r.kvco_pred / 1e6:.0f} MHz/V (c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} "
        f"max {r.vco_current_max:.0f})" for r in unlocked.itertuples()))

pd.DataFrame({
    "f_vco [MHz]": runs["f_vco"] / 1e6,
    "f_pad [MHz]": runs["f_out_meas"] / 1e6,  # analyzer centre frequency = carrier on the pad
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if not np.isnan(r.vco_tune_coarse) else "-" for r in runs.itertuples()],
    "|Kvco| target [MHz/V]": runs["kvco_target"] / 1e6,
    "|Kvco| pred [MHz/V]": runs["kvco_pred"] / 1e6,
    "Vctrl pred [V]": runs["vctrl_pred"],
    "f_vco meas [MHz]": runs["f_vco_meas"] / 1e6,
    "N": runs["total_div"],
    "I [mA]": runs["pll_i"] * 1e3,
    "P [mW]": runs["pll_p"] * 1e3,
    "carrier [dBm]": runs["carrier_dbm"],
    "jitter markers [ps]": runs["time_jitter_ps"],
    "jitter curve [ps]": runs["time_jitter_trace_ps"],
    "jitter noise [ps]": runs["time_jitter_noise_ps"],
    "jitter spurs [ps]": runs["time_jitter_spur_ps"],
    "spurs": runs["spurs"],
    "status": runs["status"],
}).round(3)

pll_spectrum_kvco_S5_20261005_171019.csv: 8 settings, 5 locked (5 measured ok); traces: pll_spectrum_kvco_S5_20261005_171019_traces.csv
excluded (not locked): |Kvco| 62 MHz/V (c10 min 0 max 10), |Kvco| 96 MHz/V (c10 min 0 max 0), |Kvco| 21142 MHz/V (c2 min 14 max 0)


,f_vco [MHz],setting,|Kvco| target [MHz/V],|Kvco| pred [MHz/V],Vctrl pred [V],f_vco meas [MHz],N,I [mA],P [mW],carrier [dBm],jitter markers [ps],jitter curve [ps],jitter noise [ps],jitter spurs [ps],spurs,status
0,1000.0,c5 min 8 max 6,328.942,460.975,0.661,999.999,4,0.607,0.455,-10.276,1437.230,2019.235,1989.711,344.033,4815 Hz: -30.9 dBc; 717000 Hz: -7.6 dBc; 70350...,ok
1,1000.0,c5 min 8 max 0,756.354,772.328,0.677,999.999,4,0.607,0.455,-10.552,1458.453,2080.902,2046.778,375.307,4830 Hz: -29.1 dBc; 52950 Hz: -18.7 dBc; 71700...,ok
2,1000.0,c9 min 9 max 10,1739.124,1720.680,0.539,999.999,4,0.723,0.542,-1.069,239.237,343.564,341.157,40.596,4830 Hz: -40.4 dBc; 5190 Hz: -49.1 dBc; 9645 H...,ok
3,1000.0,c8 min 9 max 10,3998.857,4021.177,0.567,999.999,4,0.688,0.516,-2.178,342.110,503.012,498.423,67.791,4830 Hz: -30.8 dBc; 5190 Hz: -39.5 dBc; 9645 H...,ok
4,1000.0,c6 min 11 max 0,9194.779,9231.979,0.592,999.999,4,0.630,0.473,-1.456,203.483,395.860,361.063,162.292,4815 Hz: -42.6 dBc; 9630 Hz: -45.5 dBc; 385500...,ok


In [ ]:
INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))

# One colour per setting, in run order (not cycled: more than 8 settings -> split the run).
KEYS = list(zip(runs["f_vco"], runs["kvco_target"]))
COLORS = {k: PALETTE[i] for i, k in enumerate(KEYS)}
NAMES = {(r.f_vco, r.kvco_target): f"{r.f_vco / 1e6:g} MHz, |Kvco| {r.kvco_pred / 1e6:.0f} MHz/V "
                                   f"(c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}), carrier {r.carrier_dbm:.1f} dBm"
         for r in runs.itertuples()}
N_DIV = {(r.f_vco, r.kvco_target): r.total_div for r in runs.itertuples()}
REF_TXT = (f"reference {runs['f_ref'].iloc[0] / 1e6:.4g} MHz" if runs["f_ref"].nunique() == 1  # input of the PLL
           else "reference per setting (see the table)")


def trace_of(key, kind):
    return traces[(traces["f_vco"] == key[0]) & (traces["kvco_target"] == key[1]) & (traces["kind"] == kind)]


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [ ]:
# Averaged wide spectrum around the carrier: level relative to the carrier (trace maximum), in the trace's RBW.
fig = go.Figure()
for key in KEYS:
    d = trace_of(key, "spectrum")
    if d.empty:
        continue
    fig.add_scatter(x=(d["freq"] - d["center"]) / 1e6, y=d["level_dbm"] - d["level_dbm"].max(), mode="lines",
                    line=dict(color=COLORS[key], width=1.5), name=NAMES[key],
                    hovertemplate="%{x:.4f} MHz: %{y:.1f} dBc<extra></extra>")
rbw = traces.loc[traces["kind"] == "spectrum", "rbw"]
fig.update_layout(title=f"Spectrum on the pad (= PLL output) around the carrier (RBW {rbw.iloc[0]:g} Hz), {REF_TXT}" if len(rbw) else f"Spectrum on the pad (= PLL output), {REF_TXT}",
                  template="plotly_white", height=500, legend=dict(orientation="h", y=-0.2),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier [MHz]", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261005_171019_spectrum.html and .png


In [ ]:
# VCO-referred phase noise: the full curve (thick), its noise-only version with the spurs replaced by the local floor
# (dashed), the detected spurs (x, hover: spur power ON THE PAD in dBc: strong spurs are coupled in after the divider,
# so +20 log10 N does not apply to them), each span normalized to 1 Hz (thin, upper sideband, no ENBW correction:
# shows whether the spans line up), and the phase-noise markers (dots).
fig = go.Figure()
for key in KEYS:
    n_db = 20 * np.log10(N_DIV[key])
    for span, d in trace_of(key, "phase_noise").groupby("span"):
        o = d["freq"] - d["center"]
        keep = o > 10 * d["rbw"]
        fig.add_scatter(x=o[keep], y=(d["level_dbm"] - d["level_dbm"].max() - 10 * np.log10(d["rbw"]))[keep] + n_db,
                        mode="lines", line=dict(color=COLORS[key], width=0.8), opacity=0.35, showlegend=False,
                        legendgroup=NAMES[key],
                        hovertemplate=f"span {span:g} Hz<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
    lt, ln = trace_of(key, "L_trace"), trace_of(key, "L_noise")
    if not lt.empty:
        fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + n_db, mode="lines", line=dict(color=COLORS[key], width=2.5),
                        name=NAMES[key], legendgroup=NAMES[key],
                        hovertemplate="full curve: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    if not ln.empty:
        fig.add_scatter(x=ln["freq"], y=ln["level_dbm"] + n_db, mode="lines", showlegend=False, legendgroup=NAMES[key],
                        line=dict(color=INK, width=1.2, dash="dash"),
                        hovertemplate="noise only: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
    sp = trace_of(key, "spur")
    if not sp.empty and not lt.empty:
        peak = np.interp(sp["freq"], lt["freq"], lt["level_dbm"]) + n_db
        fig.add_scatter(x=sp["freq"], y=peak, mode="markers", showlegend=False, legendgroup=NAMES[key],
                        marker=dict(symbol="x", size=11, color=COLORS[key], line=dict(width=1)),
                        customdata=sp["level_dbm"],
                        hovertemplate="spur at %{x:.4s}Hz: %{customdata:.1f} dBc on the pad<extra></extra>")
    r = runs[(runs["f_vco"] == key[0]) & (runs["kvco_target"] == key[1])].iloc[0]
    fig.add_scatter(x=[float(o) for o in offsets], y=[r[f"L_vco_{o}"] for o in offsets], mode="markers",
                    marker=dict(color=COLORS[key], size=10, line=dict(color="white", width=1)), showlegend=False,
                    legendgroup=NAMES[key], hovertemplate="marker: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title=f"Phase noise referred to the VCO, pad + 20 log10 N (thick: full curve, dashed: noise only, x: spurs, dots: markers), {REF_TXT}",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.18),
                  hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **OFFSET_AXIS)
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261005_171019_phase_noise.html and .png


In [ ]:
# RMS jitter (integration band of the run) per setting: the full curve (total, noise only, spur part) and the
# markers. x axis: f_vco when the run sweeps the frequency, else the |Kvco| of the chosen setting.
ok = runs[runs["status"] == "ok"]
SWEEP_F = ok["f_vco"].nunique() > 1
X_COL, X_SCALE, X_TITLE = (("f_vco", 1e6, "f_vco [MHz]") if SWEEP_F else
                           ("kvco_pred", 1e6, "|Kvco| of the chosen setting [MHz/V]"))
ok = ok.sort_values(X_COL)
HOVER_DATA = np.c_[ok["vco_tune_coarse"], ok["vco_current_min"], ok["vco_current_max"], ok["kvco_pred"] / 1e6,
                   ok["carrier_dbm"]]
HOVER_SETTING = ("c%{customdata[0]:.0f} min %{customdata[1]:.0f} max %{customdata[2]:.0f}, "
                 "|Kvco| %{customdata[3]:.0f} MHz/V, carrier %{customdata[4]:.1f} dBm")
fig = go.Figure()
for col, name, color, dash in (("time_jitter_trace_ps", "full curve, total", PALETTE[0], "solid"),
                               ("time_jitter_noise_ps", "full curve, noise only", PALETTE[2], "solid"),
                               ("time_jitter_spur_ps", "full curve, spurs", PALETTE[3], "dot"),
                               ("time_jitter_ps", "markers", PALETTE[1], "dash")):
    fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok[col], mode="lines+markers", name=name,
                    line=dict(color=color, width=2, dash=dash), marker=dict(size=9), customdata=HOVER_DATA,
                    hovertemplate=f"{name}: %{{y:.1f}} ps at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(title=f"RMS time jitter (same on the pad and at the VCO), integrated {float(offsets[0]):g} Hz - {float(offsets[-1]) / 1e6:g} MHz, {REF_TXT}",
                  template="plotly_white", height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title=X_TITLE, type="log", gridcolor=GRID)
fig.update_yaxes(title="RMS time jitter [ps]", type="log", gridcolor=GRID)
save(fig, "jitter_vs_f" if SWEEP_F else "jitter_vs_kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261005_171019_jitter_vs_kvco.html and .png


In [ ]:
# PLL supply power and carrier level on the pad per setting (same x axis as the jitter plot).
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["PLL supply power", "Carrier level on the pad (= PLL output)"])
fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok["pll_p"] * 1e3, mode="lines+markers", showlegend=False, row=1, col=1,
                line=dict(color=PALETTE[0], width=2), marker=dict(size=9), customdata=HOVER_DATA,
                hovertemplate=f"%{{y:.3f}} mW at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.add_scatter(x=ok[X_COL] / X_SCALE, y=ok["carrier_dbm"], mode="lines+markers", showlegend=False, row=2, col=1,
                line=dict(color=PALETTE[1], width=2), marker=dict(size=9), customdata=HOVER_DATA,
                hovertemplate=f"%{{y:.1f}} dBm at %{{x:.0f}}<br>{HOVER_SETTING}<extra></extra>")
fig.update_layout(title=f"PLL supply power and carrier level on the pad, {REF_TXT}", template="plotly_white", height=600)
fig.update_xaxes(type="log", gridcolor=GRID)
fig.update_xaxes(title=X_TITLE, row=2)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="power [mW]", rangemode="tozero", row=1)
fig.update_yaxes(title="carrier [dBm]", row=2)
save(fig, "power_carrier_vs_f" if SWEEP_F else "power_carrier_vs_kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_kvco_S5_20261005_171019_power_carrier_vs_kvco.html and .png
